# 🧠 Policy RAG Chatbot
### From LLM Fundamentals → A Working Multi-Organization Policy Assistant
**Stack:** Python + LangChain + Google Gemini + Gradio

---

**What we're building:** a small RAG chatbot that answers questions **strictly** from uploaded policy PDFs — one or more organizations at a time — and refuses to answer (or invent) anything that isn't actually in the retrieved policy text. Every answer comes with a citation back to the source PDF, page, and organization.

> 👋 **Not a Python expert? That's fine.** Sections 1–6 build up the underlying concepts (frameworks, memory, embeddings, RAG) with small, explained examples. Section 7 onward is the real project.

### Roadmap
1. Introduction to AI Frameworks
2. Configure the LLM (Gemini via LangChain)
3. LLM Parameters (temperature, max tokens, model choice)
4. Build a Basic Chatbot
5. Conversation History & Memory
6. Introduce RAG (Retrieval-Augmented Generation)
7. **Project:** Build the Policy RAG Chatbot core (PDF ingestion, organization tagging, strict retrieval, citations)
8. **Project:** Gradio App — organization/LLM dropdowns, PDF upload, chat

**Scope note (V1, kept intentionally simple):** PDF policies only, one active policy per organization (uploading a new PDF for an organization replaces its previous policy — this is our simple stand-in for "latest policy only"), organization and policy version/name are entered manually at upload time instead of being auto-detected, vectors are kept in-memory for the notebook session (no Qdrant Cloud), and conversation history is **not** part of this version. These match the "Do later" list in the project plan and can be layered on top of this notebook later.

**How to work through this notebook:** run each code cell in order (Shift+Enter). Nothing here needs internet access besides calling the Gemini API.

## 1. Introduction to AI Frameworks

**What is an AI/LLM framework?**
Underneath every "AI feature" is really just an HTTP call to a model: you send text in, you get text back. A **framework** is a library that sits on top of that raw API call and gives you reusable building blocks for the patterns you'll use *every single time* — sending prompts, keeping conversation state, chunking and searching documents, calling tools/functions, chaining multiple steps together.

**Why bother with a framework instead of calling the raw API yourself?**
- Less boilerplate — prompt templates, retries, output parsing already done for you.
- Swappable models — same code structure works whether the model behind it is Gemini, GPT, Claude, or a local model.
- Built-in patterns for the exact things we're about to build: memory, retrieval (RAG), agents/tools.
- Large community, docs, and pre-built integrations (vector databases, document loaders, etc).

**Popular frameworks (this concept exists in every language):**

| Framework | Primary language(s) | Notes |
|---|---|---|
| **LangChain** | Python, JavaScript/TypeScript | Most widely adopted, general-purpose orchestration |
| **LlamaIndex** | Python, TypeScript | Strong focus on data/RAG pipelines |
| **Semantic Kernel** | C#, Python, Java | Microsoft's framework, popular in .NET shops |
| **LangChain4j / Spring AI** | Java | LangChain-style concepts for the JVM |
| **Vercel AI SDK** | JavaScript/TypeScript | Popular for web app front-ends |
| Provider SDKs (`google-generativeai`, `openai`, `anthropic`) | Every major language | Lower-level, no orchestration built in |

**Where does LangChain fit?** It's a general-purpose orchestration layer: prompt templates → memory → retrieval → tools/agents, all behind one consistent interface, with adapters for almost every LLM provider (including Gemini, which we'll use today).

**Why Python + LangChain for this workshop?**
- Python is the de-facto language of the AI/ML ecosystem — most tutorials, papers, and new integrations show up here first.
- LangChain is the most mature and most documented of these frameworks, so it's the fastest way to *see the concepts in action*.
- Zero local setup — this entire notebook runs in Google Colab in a browser.
- Every concept you learn today (frameworks, memory, embeddings, RAG) maps 1:1 onto whatever language/framework your own team uses — only the syntax changes.

We're deliberately keeping the LangChain usage **minimal**. The goal is that you understand *what a chatbot/RAG system is actually doing*, not that you memorize a framework's API surface.


## 2. Configure the LLM

We'll now go from zero to "talking to Gemini" in four steps:
1. Install the required packages.
2. Provide your Gemini API key **securely** (never hardcoded, never saved into this notebook file).
3. Initialize the Gemini model through LangChain.
4. Make our first LLM call.

### 2.1 Install packages


In [ ]:
# LangChain core + the Gemini integration + Google's own SDK (used for a couple of direct calls later)
# pypdf/langchain-community: PDF loading. gradio: the demo UI for the Policy RAG Chatbot.
!pip install -q langchain langchain-core langchain-community langchain-google-genai google-generativeai langchain-text-splitters pypdf gradio

`langchain-core` gives us the framework's core building blocks (messages, prompts, vector store interfaces). `langchain-google-genai` is the adapter that lets LangChain talk to Gemini. `google-generativeai` is Google's own SDK — LangChain uses it under the hood, and we'll use it directly once to *list available models*.

### 2.2 Provide your Gemini API key (securely)

Never write an API key directly into a notebook cell — if you share the notebook or push it to a repo, the key leaks with it. Instead we'll use `getpass`, which prompts for input **without echoing it to the screen**, and store it only in this runtime's memory (as an environment variable) — it disappears when the Colab runtime is reset.

> Don't have a key yet? Get one for free at **[Google AI Studio](https://aistudio.google.com/app/apikey)**.


In [ ]:
import os
from getpass import getpass

# You will be prompted to paste your key below. It will NOT be displayed or saved to the notebook.
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

print("API key loaded into this session ✅ (not printed, not saved to the notebook file)")


API key loaded into this session ✅ (not printed, not saved to the notebook file)


### 2.3 Initialize the Gemini model through LangChain

`ChatGoogleGenerativeAI` is a LangChain **wrapper class** around Gemini's chat API. Once wrapped, it behaves the same way every other LangChain chat model does — this is exactly the "swappable model" benefit we mentioned in Section 1.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",  # a fast, low-cost Gemini model — good default for a workshop
    temperature=0.7,                # we'll explain this in Section 3
)

print("Model initialized:", llm.model)


Model initialized: gemini-3.1-flash-lite


### 2.4 Your first LLM call


In [ ]:
response = llm.invoke("In one sentence, explain what a Large Language Model is.")
print(response.content)


[{'type': 'text', 'text': 'A Large Language Model is a type of artificial intelligence trained on vast amounts of text data to understand, generate, and predict human language patterns.', 'extras': {'signature': 'EnEKbwERTTIP3tnOg2Bx+r8bGjGRHwP+TagQGIgt3CDMY98arhba5rQ8P7kBUdwUeoXNXR6ZWyVhZQnfXi1fbgyC82vJZNW880Nz3VYht070m+eni/i7s/SPU6V+8R3dZECOns+6iIMnGC/UeAJpl3oLUQ=='}}]


**What just happened behind the scenes?**

1. `llm.invoke("...")` took your plain string and wrapped it into a request Gemini understands.
2. LangChain sent that request over HTTPS to Google's Gemini endpoint, authenticated using your API key.
3. Gemini generated a reply token-by-token on Google's servers.
4. The full reply came back as an `AIMessage` object — `response`.

**One quirk to flag before we move on:** `response.content` is *usually* a plain string — but depending on the model, it can also come back as a **list of content blocks** instead. Look at what the cell above printed: if it was a plain sentence, you got a string; if it looked like `[{'type': 'text', 'text': '...', 'extras': {...}}]`, you got the list form (some Gemini 3 models attach an internal "thought signature" block alongside the visible text — that's the `extras` part, and we don't care about it).

Either way, all we ever want is the visible text. So instead of writing `.content` everywhere and handling both cases every time, let's write one tiny helper *once* and reuse it for every response for the rest of the workshop — that keeps the comparisons in Section 3 (and everything after) easy to read.


In [ ]:
def get_text(response):
    """Return just the plain text of an LLM response, whether `.content` is a string
    or a list of content blocks (e.g. text + an internal thought-signature block)."""
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )

print(get_text(response))


A Large Language Model is a type of artificial intelligence trained on vast amounts of text data to understand, generate, and predict human language patterns.


**Important:** that call was completely self-contained. Gemini does not remember you asked it anything — it has no memory between calls. Keep that in mind; it's the whole reason Section 5 exists.


## 3. LLM Parameters

Every LLM call is controlled by a handful of parameters. The three that matter most day-to-day:

| Parameter | What it controls | Typical range |
|---|---|---|
| `temperature` | Randomness / "creativity" of the output | `0.0` (deterministic) → `1.0`+ (more random) |
| `max_output_tokens` | Hard cap on how long the response can be | depends on model, e.g. 1–8192 |
| `model` | Which underlying model answers — trades off speed/cost vs. capability | e.g. `gemini-3.1-flash-lite` vs `gemini-3.5-flash` |

Let's *see* the effect of each one instead of just reading about it.

### 3.1 Temperature: low vs. high, same prompt


In [ ]:
prompt = "Give me a one-sentence tagline for a coffee shop."

llm_low_temp = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)
llm_high_temp = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=1.0)

print("=== temperature = 0.0 (run 3 times) ===")
for i in range(3):
    print(f"{i+1}.", get_text(llm_low_temp.invoke(prompt)))

print("\n=== temperature = 1.0 (run 3 times) ===")
for i in range(3):
    print(f"{i+1}.", get_text(llm_high_temp.invoke(prompt)))


=== temperature = 0.0 (run 3 times) ===
1. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*   **For a high-energy vibe:** "Fueling your ambition, one pour at a time."
*   **For a quality-focused vibe:** "Crafted with passion, poured to perfection."
*   **For a community-focused vibe:** "Where great coffee meets good company."
*   **Short and punchy:** "Awaken your senses."
2. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*   **For a high-energy vibe:** "Fueling your ambition, one pour at a time."
*   **For a quality-focused vibe:** "Crafted with passion, poured to perfection."
*   **For a community-focused vibe:** "Where great coffee meets good company."
*   **Short and punchy:** "Awaken your senses."
3. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*

**Expected behavior:** the `temperature=0.0` runs should come back nearly identical (or exactly identical) each time — the model is picking the single most-likely next word every time. The `temperature=1.0` runs should vary more from each other — the model is now sometimes picking less-likely (but still plausible) words, trading consistency for variety.

**When to use which:** low temperature for factual/deterministic tasks (data extraction, classification, code generation), higher temperature for creative tasks (brainstorming, marketing copy, varied phrasing).

### 3.2 Maximum output tokens

Tokens are roughly "chunks of text" (not quite words, not quite characters). `max_output_tokens` puts a hard ceiling on the length of the response — useful for controlling cost/latency, or for forcing short answers.


In [ ]:
prompt = "Explain how a vector database works."

llm_short = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3, max_output_tokens=20)
llm_long = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3, max_output_tokens=300)

print("=== max_output_tokens = 20 ===")
print(get_text(llm_short.invoke(prompt)))

print("\n=== max_output_tokens = 300 ===")
print(get_text(llm_long.invoke(prompt)))


=== max_output_tokens = 20 ===
To understand how a vector database works, you first need to understand the concept of

=== max_output_tokens = 300 ===
To understand how a vector database works, you first have to understand the core concept: **Vector Embeddings.**

In traditional databases (SQL), you search for data using exact matches (e.g., "Find the user with ID 123"). In a vector database, you search for **meaning** (e.g., "Find documents that are conceptually similar to this query").

Here is the step-by-step breakdown of how a vector database functions.

---

### 1. The Foundation: Vector Embeddings
Computers cannot understand the meaning of a photo, a paragraph of text, or an audio file. To bridge this gap, we use **Machine Learning models** (like OpenAI’s `text-embedding-3` or CLIP for images) to convert raw data into a **Vector Embedding**.

*   **What is a vector?** It is simply a long list of numbers (e.g., `[0.12, -0.59, 0.88, ...]`).
*   **The Magic:** These numbers represe

**Expected behavior:** the first response should be cut off mid-thought — it hit the token ceiling before finishing. The second should be a complete explanation. This is why real applications set a sensible `max_output_tokens`: an unbounded response can be slow and expensive, but too small a cap gives truncated, unusable answers.

### 3.3 Model selection

Gemini (like most providers) offers several model sizes — smaller/faster/cheaper vs. larger/slower/more capable. Which one is "current" changes over time, so rather than trusting a hardcoded name, let's ask the API what's actually available right now.


In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Models available to your key that support chat:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(" -", m.name)


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Models available to your key that support chat:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - models/lyria-3-clip-preview
 - models/lyria-3-pro-preview
 - mode

We'll compare our default `gemini-3.1-flash-lite` (lighter/cheaper) against `gemini-3.5-flash` (fuller model) on the same prompt — confirm both names appear in the list printed above for your key/region before running the next cell.


In [ ]:
import time

prompt = "Explain the difference between Agentic AI and RAG in 2 sentences."

for model_name in ["gemini-3.1-flash-lite", "gemini-3.5-flash"]:
    fast_llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.3)
    start = time.time()
    result = fast_llm.invoke(prompt)
    elapsed = time.time() - start
    print(f"--- {model_name} ({elapsed:.1f}s) ---")
    print(get_text(result))
    print()


--- gemini-3.1-flash-lite (8.4s) ---
RAG (Retrieval-Augmented Generation) is a technique that provides an AI with external data to improve the accuracy of its responses to specific queries. In contrast, Agentic AI refers to autonomous systems capable of using tools, reasoning through multi-step plans, and taking independent actions to achieve complex goals.

--- gemini-3.5-flash (10.0s) ---
**RAG (Retrieval-Augmented Generation)** is a technique that improves an AI's answers by fetching relevant external data to ground its responses in specific facts. In contrast, **Agentic AI** is an autonomous system designed to proactively plan, make decisions, use tools, and execute multi-step workflows to achieve complex goals.



**What to notice:** `gemini-3.1-flash-lite` usually answers noticeably faster, while `gemini-3.5-flash` tends to give a more thorough/nuanced answer — that speed-vs-capability tradeoff is the main thing you're choosing when you pick a model. In production you'd pick based on your actual requirements (latency budget, cost budget, task difficulty) — not just "use the best model available."

> If either model name above errors out for your key/region, re-run the previous cell and copy an exact name from the printed list.


## 4. Build a Basic Chatbot

The simplest possible "chatbot" is just:

```
User → LLM → Response
```

No memory, no history — every message is a fresh, independent call. Let's build that and immediately see its limitation.


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.7)

def chat(user_message):
    response = llm.invoke(user_message)
    return get_text(response)

print(chat("Hi! My name is Alex."))


Hi Alex! It’s great to meet you. How are you doing today? Is there anything I can help you with?


In [ ]:
print(chat("What is my name?"))


I don’t know your name. As an AI, I don’t have access to your personal identity, documents, or private information unless you have previously shared it with me in this specific conversation.


**Expected behavior:** the model has no idea what your name is — it will say it doesn't know, or ask you to tell it. This isn't a bug. `chat()` calls `llm.invoke(...)` fresh each time, with *only* the new message. The previous exchange was never sent along, so as far as Gemini is concerned, this second call is the start of a brand-new conversation.

**This is true of essentially all LLM APIs**, not just Gemini: the model itself is stateless. Any "memory" you experience in a chat product (like the Gemini or ChatGPT web apps) is the *application* resending the conversation so far — which is exactly what we'll build next.


## 5. Conversation History & Memory

To make a chatbot feel continuous, the application has to keep track of what's been said and **resend it** on every call. Let's define a few terms precisely, because they get used loosely in practice:

- **Chat history** — the literal ordered list of messages exchanged so far (`Human: ...`, `AI: ...`, ...). This is the raw transcript.
- **Session** — one logical, continuous conversation instance (often identified by a session/conversation ID) that a chat history belongs to. A user might have many sessions over time.
- **Short-term memory** — the (usually recent) chat history that's actively resent to the model so it has context for the *current* session. This is what we're about to build.
- **Long-term memory** — information deliberately extracted and *persisted beyond a single session* (e.g. saved to a database or vector store) so it can be recalled in a future, separate session — "remembers you're vegetarian" three weeks later, not just three messages ago.

Today we're building **short-term memory** — a simple in-memory list of messages, kept only for the life of this notebook session.

### 5.1 LangChain's message types

LangChain represents a conversation as a list of typed messages:
- `SystemMessage` — instructions for how the model should behave.
- `HumanMessage` — something the user said.
- `AIMessage` — something the model said.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

history = [SystemMessage(content="You are a friendly, concise assistant.")]

def chat_with_history(user_message):
    history.append(HumanMessage(content=user_message))
    response = llm.invoke(history)          # send the WHOLE conversation so far
    reply_text = get_text(response)
    history.append(AIMessage(content=reply_text))
    return reply_text

print(chat_with_history("My name is Alex."))


Hi Alex! It's nice to meet you. How can I help you today?


In [ ]:
print(chat_with_history("What is my name?"))


Your name is Alex!


**Expected behavior:** this time the model correctly answers "Alex" — because the full `history` list (system prompt + every human/AI turn so far) is sent to the model on *every single call*. Nothing is stored on Google's servers between calls; **we** are resending the transcript each time, and the model re-reads it from scratch.

Run the cell below to see exactly what's being sent under the hood:


In [ ]:
for msg in history:
    print(f"[{msg.type}] {msg.content}")


[system] You are a friendly, concise assistant.
[human] My name is Alex.
[ai] Hi Alex! It's nice to meet you. How can I help you today?
[human] What is my name?
[ai] Your name is Alex!


**A practical consequence:** the longer a conversation gets, the more text you resend (and pay for/wait on) with every single turn — this is why real chat products eventually summarize or trim older history instead of keeping it forever. We won't build that today, but it's worth knowing it's the next problem you'd hit.

**Recap of the four terms:**

| Term | What it is | Lifespan |
|---|---|---|
| Chat history | The raw list of messages | As long as you keep the list |
| Session | The conversation instance the history belongs to | Usually one user visit/interaction |
| Short-term (working) memory | History actively resent to the model for context | Current session only |
| Long-term memory | Facts deliberately saved for future sessions | Persists across sessions (needs a DB/vector store) |


## 6. Introduce RAG (Retrieval-Augmented Generation)

**Why not just paste the whole document into the prompt?**
- LLMs have a limited **context window** — very large or many documents simply won't fit.
- Even when it fits, sending a huge document on *every* question is slow and expensive (you pay/wait per token, every single call).
- Irrelevant surrounding text can distract the model and increase the chance of a wrong or hallucinated answer — needle-in-a-haystack problems are real.
- Real knowledge bases (wikis, ticket histories, codebases) are usually far bigger than any context window anyway.

**The idea behind RAG:** instead of sending the whole document, *search it first* for the few pieces that are actually relevant to the current question, and send only those pieces to the LLM.

```
Document
   ↓
Load
   ↓
Chunk
   ↓
Embedding
   ↓
Vector Database
   ↓
Similarity Search
   ↓
Relevant Chunks
   ↓
LLM
   ↓
Answer
```

**What each step means:**
- **Document** — your raw source of knowledge (a PDF, wiki page, text file, ...).
- **Load** — read it into a standard in-memory representation the framework understands.
- **Chunk** — split it into smaller, overlapping pieces. Smaller pieces mean each one is about *one specific topic*, so a search against them is more precise than searching whole documents.
- **Embedding** — convert each chunk of text into a vector (a list of numbers) that captures its *meaning*. Texts with similar meaning end up with similar vectors, even if they don't share exact words.
- **Vector database** — a store optimized for holding these vectors and quickly finding the ones closest to a given query vector.
- **Similarity search** — embed the user's *question* the same way, then find the chunks whose vectors are closest to it — i.e. the most semantically relevant chunks.
- **Relevant chunks** — the small handful of chunks that actually matter for this question (out of possibly thousands).
- **LLM** — given the question *plus* those relevant chunks as context, generate an answer grounded in the real document.
- **Answer** — the final response, ideally only using facts that were actually in the retrieved chunks.

Let's build this end-to-end with a tiny sample document.


## 7. Build the Policy RAG Chatbot — Core Pipeline

This section builds the real pipeline from the project plan:

```
PDF Upload → Page-aware Text Extraction → Organization/Policy Tagging → Chunking
   → Embeddings → In-memory Vector Store → Organization-filtered Retrieval
   → Strict LLM Prompt → Answer + Citations (or the required "no answer" message)
```

The key idea that makes this **multi-organization**: every chunk we store carries metadata (`organization`, `policy_name`, `policy_version`, `source_file`, `page`). At question time we filter the vector search down to only the chunks belonging to the organization selected in the UI — so Company A's policies can never leak into an answer for Company B.

### 7.1 PDF loading (page-aware)

`PyPDFLoader` reads a PDF and returns **one `Document` per page**, each already carrying a `page` number in its metadata. We wrap that loader in a small function that also stamps on our own organization/policy metadata (entered manually for this simple version, instead of being auto-detected from the PDF).

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_as_documents(pdf_path, organization, policy_name, policy_version):
    """Load a PDF page-by-page and tag every page with organization/policy metadata."""
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()  # one Document per page, metadata already has "page"

    for page_doc in pages:
        page_doc.metadata.update({
            "organization": organization,
            "policy_name": policy_name,
            "policy_version": policy_version,
            "source_file": os.path.basename(pdf_path),
        })
    return pages

print("load_pdf_as_documents() ready — we'll use it once we have a PDF to try in Section 8.")

### 7.2 Chunking (metadata-preserving)

Same `RecursiveCharacterTextSplitter` idea as before — but now `split_documents` runs per-page `Document`, so every resulting chunk automatically **keeps** the `organization`/`policy_name`/`page` metadata of the page it came from. That metadata is what makes citations possible later.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

policy_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

def chunk_documents(page_documents):
    return policy_splitter.split_documents(page_documents)

print("chunk_documents() ready.")

### 7.3 Embeddings + a single shared vector store

We use **one** vector store for every organization (not one store per org) — the multi-tenant separation happens through metadata filtering at query time (Section 7.4), not through separate stores. This is simpler to manage and is exactly how a metadata-filtered collection in a real vector DB (like Qdrant, per the plan) would work too.

`index_policy_pdf(...)` ties Sections 7.1–7.3 together: load → tag → chunk → embed → store. It also implements our simple **"latest policy only"** rule: before adding the new chunks, it deletes any chunks already stored for that organization, so only the most recently uploaded policy is ever retrievable per organization.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
embedding_model_name = next(
    m.name for m in genai.list_models() if "embedContent" in m.supported_generation_methods
)
print("Using embedding model:", embedding_model_name)

embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model_name)
policy_vector_store = InMemoryVectorStore(embeddings)

# Track which organizations currently have an indexed policy (drives the UI dropdown in Section 8)
indexed_organizations = {}  # organization -> {"policy_name", "policy_version", "source_file", "chunk_count"}

def index_policy_pdf(pdf_path, organization, policy_name, policy_version):
    """Load, tag, chunk, embed, and store a policy PDF. Replaces any previous policy for this org."""
    # Enforce "latest policy only": drop whatever was previously indexed for this organization
    existing_ids = [
        doc_id for doc_id, doc in policy_vector_store.store.items()
        if doc.get("metadata", {}).get("organization") == organization
    ]
    if existing_ids:
        policy_vector_store.delete(ids=existing_ids)

    pages = load_pdf_as_documents(pdf_path, organization, policy_name, policy_version)
    chunks = chunk_documents(pages)
    policy_vector_store.add_documents(chunks)

    indexed_organizations[organization] = {
        "policy_name": policy_name,
        "policy_version": policy_version,
        "source_file": os.path.basename(pdf_path),
        "chunk_count": len(chunks),
    }
    return len(chunks)

print("index_policy_pdf() ready.")

### 7.4 Organization-filtered retrieval

`InMemoryVectorStore.similarity_search` accepts a `filter` — a function run against each candidate `Document`'s metadata. We use it to restrict semantic search to chunks whose `organization` matches the one selected in the UI, **before** ranking by similarity. This is the line that guarantees Company A's policy content can never answer a question asked under Company B.

In [ ]:
def retrieve_for_organization(question, organization, k=4):
    return policy_vector_store.similarity_search(
        question,
        k=k,
        filter=lambda doc: doc.metadata.get("organization") == organization,
    )

print("retrieve_for_organization() ready.")

### 7.5 Strict policy-only prompt, citations, and the required no-answer message

Straight from the project plan's answering rules:
- Answer **only** from retrieved context — no outside knowledge, no inference.
- Every supported answer must carry a citation (policy name, version, source file, page).
- If the retrieved context doesn't support an answer, return **exactly**: *"I couldn't find this information in the available policies."*

Citations are built from the retrieved chunks' own metadata — never invented by the LLM.

In [ ]:
NO_ANSWER_MESSAGE = "I couldn't find this information in the available policies."

STRICT_POLICY_PROMPT = """You are a policy assistant. Answer the question using ONLY the CONTEXT below,
which comes from official policy documents for {organization}.

Rules:
- Do not use outside/general knowledge to fill in missing details.
- Do not infer a rule the context does not actually state.
- If the question is not about the policy content, or the context does not contain the answer,
  respond with EXACTLY this sentence and nothing else: "{no_answer}"

CONTEXT:
{context}

Question: {question}

Answer:"""

_llm_cache = {}

def get_llm(model_name, temperature=0.0):
    if model_name not in _llm_cache:
        _llm_cache[model_name] = ChatGoogleGenerativeAI(model=model_name, temperature=temperature)
    return _llm_cache[model_name]

def format_citation(doc):
    meta = doc.metadata
    return (f"{meta.get('organization', '?')} — {meta.get('policy_name', '?')} "
            f"{meta.get('policy_version', '')} (page {meta.get('page', '?')}, "
            f"{meta.get('source_file', '?')})").strip()

def ask_policy_question(question, organization, model_name):
    """Returns (answer_text, citations_list). Strictly grounded in retrieved policy context."""
    if organization not in indexed_organizations:
        return NO_ANSWER_MESSAGE, []

    retrieved_docs = retrieve_for_organization(question, organization)
    if not retrieved_docs:
        return NO_ANSWER_MESSAGE, []

    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = STRICT_POLICY_PROMPT.format(
        organization=organization, no_answer=NO_ANSWER_MESSAGE, context=context, question=question,
    )

    llm_for_answer = get_llm(model_name)
    answer_text = get_text(llm_for_answer.invoke(prompt)).strip()

    if answer_text == NO_ANSWER_MESSAGE:
        return answer_text, []

    # Citations always come from the retrieved chunks' own metadata, never from the LLM's output.
    citations = sorted({format_citation(doc) for doc in retrieved_docs})
    return answer_text, citations

print("ask_policy_question() ready.")

## 8. Gradio App — Policy Assistant UI

This wires Section 7's pipeline into the UI described in the project plan:

- **Organization** dropdown — refreshes automatically as PDFs get indexed. Testing control that also enforces isolation (retrieval is filtered to whatever is selected).
- **LLM Model** dropdown — swap models and re-ask the same question, for comparison.
- **Upload Policy PDF** — organization name, policy name, and policy version are entered manually (our simplified stand-in for auto-detection), then indexed with one click.
- **Policy status** — shows what's currently indexed for the selected organization.
- **Chat box** — strict, grounded answers with citations, or the exact fallback message when the answer isn't supported.

In [ ]:
import gradio as gr

# A small, curated set of chat-capable models for the dropdown (avoids preview/experimental clutter)
_preferred_models = ["gemini-3.1-flash-lite", "gemini-3.5-flash", "gemini-2.5-flash", "gemini-2.5-pro"]
_available_model_names = {m.name.split("/")[-1] for m in genai.list_models() if "generateContent" in m.supported_generation_methods}
available_models = [m for m in _preferred_models if m in _available_model_names] or list(_available_model_names)[:4]

def do_index(pdf_file, organization, policy_name_, policy_version_):
    if not pdf_file or not organization or not policy_name_:
        return gr.update(), "⚠️ Please provide a PDF, organization name, and policy name."
    organization = organization.strip()
    chunk_count = index_policy_pdf(pdf_file.name, organization, policy_name_.strip(), (policy_version_ or "n/a").strip())
    orgs = sorted(indexed_organizations.keys())
    status = f"✅ Indexed **{policy_name_}** for **{organization}** ({chunk_count} chunks)."
    return gr.update(choices=orgs, value=organization), status

def show_policy_status(organization):
    if not organization or organization not in indexed_organizations:
        return "_No policy indexed for this organization yet._"
    info = indexed_organizations[organization]
    return (f"**Active policy for {organization}:** {info['policy_name']} {info['policy_version']}  \n"
            f"Source: `{info['source_file']}` · {info['chunk_count']} chunks indexed")

def do_ask(question, organization, model_name, chat_history):
    chat_history = chat_history or []
    if not question.strip():
        return chat_history, ""
    if not organization:
        chat_history.append({"role": "assistant", "content": "Please select (or index) an organization first."})
        return chat_history, ""

    answer, citations = ask_policy_question(question, organization, model_name)
    if citations:
        answer += "\n\n---\n**Sources:**\n" + "\n".join(f"- {c}" for c in citations)

    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})
    return chat_history, ""

with gr.Blocks(title="Policy Assistant") as demo:
    gr.Markdown("# 📋 Policy Assistant")
    with gr.Row():
        org_dropdown = gr.Dropdown(choices=sorted(indexed_organizations.keys()), label="Organization", allow_custom_value=True)
        llm_dropdown = gr.Dropdown(choices=available_models, value=available_models[0], label="LLM Model")

    with gr.Accordion("Upload / Index a Policy PDF", open=True):
        with gr.Row():
            pdf_file = gr.File(label="Policy PDF", file_types=[".pdf"])
            org_input = gr.Textbox(label="Organization (e.g. Company A)")
        with gr.Row():
            policy_name_input = gr.Textbox(label="Policy Name (e.g. Leave Policy)")
            policy_version_input = gr.Textbox(label="Policy Version (e.g. v3.0)")
        index_btn = gr.Button("Index Policy", variant="primary")
        index_status = gr.Markdown()

    policy_status = gr.Markdown("_No policy indexed for this organization yet._")
    chatbot = gr.Chatbot(label="Policy Q&A", type="messages", height=350)
    question_box = gr.Textbox(label="Ask a question about the selected organization's policy", placeholder="e.g. How many vacation days do I get?")
    ask_btn = gr.Button("Ask")

    index_btn.click(do_index, [pdf_file, org_input, policy_name_input, policy_version_input], [org_dropdown, index_status]) \
             .then(show_policy_status, org_dropdown, policy_status)
    org_dropdown.change(show_policy_status, org_dropdown, policy_status)
    ask_btn.click(do_ask, [question_box, org_dropdown, llm_dropdown, chatbot], [chatbot, question_box])
    question_box.submit(do_ask, [question_box, org_dropdown, llm_dropdown, chatbot], [chatbot, question_box])

demo.launch(debug=False)

## Recap: what you built

```
LLM → Parameters → PDF Loading → Org/Policy Tagging → Chunking → Embeddings
   → In-Memory Vector Store → Org-Filtered Retrieval → Strict RAG Prompt → Citations → Gradio UI
```

A Policy RAG Chatbot that:
- ingests real policy **PDFs**, page by page,
- supports **multiple organizations**, and never mixes their policy content in an answer,
- keeps only the **latest** policy per organization (simplified: newest upload replaces the old one),
- answers **strictly** from retrieved policy text, with the required fallback when it can't,
- shows a **citation** (policy, version, source file, page) for every supported answer,
- lets you **A/B different LLMs** on the same question via the model dropdown.

This covers the "Build now" boundary from the project plan (Section 13): PDF ingestion, multiple organizations, latest-policy-only retrieval, strict RAG, citations, no-answer behavior, and testing dropdowns — implemented in the simplest form that satisfies each requirement.

**What was intentionally left for later** (Section 11 of the plan):
- Conversation history (persisted chats, history sidebar, search, rename/delete)
- Automatic policy metadata/version detection (effective date, filename parsing, conflict resolution) instead of manual entry
- Qdrant Cloud (or another persistent vector DB) instead of the in-memory store — needed once you want indexed policies to survive a runtime restart
- DOCX / TXT / Markdown / Web URL ingestion
- Authentication & roles, policy comparison, and an evaluation dashboard

Each of those can be added on top of this notebook's pipeline without changing its core shape.